In [ ]:
%uv pip install torchcodec==0.7
%uv pip install nemo_toolkit['all']
datasets==4.4.1 
numpy==2.1.2
pandas==2.3.2

Using Python 3.12.6 environment at: /usr/local
Resolved 36 packages in 250ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
dill       ------------------------------ 16.00 KiB/116.86 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 16.00 KiB/116.86 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 16.00 KiB/116.86 KiB
datasets   ------------------------------     0 B/499.60 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 16.00 KiB/116.86 KiB
datasets   ------------------------------ 16.00 KiB/499.60 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 16.00 KiB/116.86 KiB
multiprocess ------------------------------     0 B/146.76 KiB
datasets   ------------------------------ 16.00 KiB/499.60 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 16.00 KiB/116.86 KiB
multiprocess ------------------------------ 16.00 KiB/1

In [3]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import torch
import re
from jiwer import wer, cer
import nemo.collections.asr as nemo_asr 
import librosa   

import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")

[NeMo W 2025-11-08 20:20:12 nemo_logging:405] /usr/local/lib/python3.12/site-packages/lhotse/recipes/iwslt22_ta.py:323: SyntaxWarning: invalid escape sequence '\s'
      text = re.sub("\s+", " ", text)
    
[NeMo W 2025-11-08 20:20:12 nemo_logging:405] /usr/local/lib/python3.12/site-packages/lhotse/recipes/iwslt22_ta.py:324: SyntaxWarning: invalid escape sequence '\s'
      text = re.sub("\s+\.\s+", ".", text)
    
[NeMo W 2025-11-08 20:20:14 nemo_logging:405] /usr/local/lib/python3.12/site-packages/megatron/core/transformer/pipeline_parallel_layer_layout.py:128: SyntaxWarning: assertion is always true, perhaps remove parentheses?
      assert (
    
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
bitsandbytes library load error: Configured CUDA binary not found at

Imports successful!


In [4]:
DATASETS = {
    # "egyptian_asr": {
    #     "name": "MightyStudent/Egyptian-ASR-MGB-3",
    #     "split": "train", 
    #     "audio_col": "audio",
    #     "text_col": "sentence"
    # },
    "arabic_egy": {
        "name": "MAdel121/arabic-egy-cleaned", 
        "split": "test",
        "audio_col": "audio", 
        "text_col": "text"
    }
}

# Base Models
MODELS = {
    "nvidia-conformer": "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
def normalize_arabic_text(text):
    """
    Arabic text normalization:
    1. Remove punctuation
    2. Remove diacritics
    3. Eastern Arabic numerals to Western Arabic numerals

    Arguments
    ---------
    text: str
        text to normalize
    Output
    ---------
    normalized text
    """
    punctuation = r'[!\"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~،؛؟]'
    text = re.sub(punctuation, '', text)

    diacritics = r'[\u064B-\u0652]'  
    text = re.sub(diacritics, '', text)
    
    text = re.sub('پ', 'ب', text)
    text = re.sub('ڤ', 'ف', text)
    text = re.sub(r'[آ]', 'ا', text)
    text = re.sub(r'[أإ]', 'ا', text)
    text = re.sub(r'[ؤ]', 'و', text)
    text = re.sub(r'[ئ]', 'ي', text)
    text = re.sub(r'[ء]', '', text)   

    eastern_to_western_numerals = {
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', 
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
    }
    for eastern, western in eastern_to_western_numerals.items():
        text = text.replace(eastern, western)

    return text.strip()


def evaluate_model(model_name, dataset_config, max_samples=50):
    """
    Evaluation function that handles both Whisper and NeMo models.
    """
    
    print(f"\nTesting {model_name}...")
    
    try:
        # Load dataset
        dataset = load_dataset(dataset_config["name"], split=dataset_config["split"])
        
        if max_samples is not None and len(dataset) > max_samples:
            dataset = dataset.select(range(max_samples))
        
        print(f"Loaded {len(dataset)} samples from {dataset_config['name']}.")

        # --- Model Loading ---
        processor = None
        model = None
        asr_model = None
        
        print(f"Loading NeMo model: {model_name}")
        asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name).to(device)
        asr_model.eval()

        # ---------------------
        
        references = []
        predictions = []
        
        # Process each sample
        for sample in tqdm(dataset, desc=f"Evaluating {model_name}"):
            audio = sample[dataset_config["audio_col"]]
            refereaudio = sample[dataset_config["audio_col"]]
            reference_text = sample[dataset_config["text_col"]]

            # Error here as audio_array not return array 
            audio_array = audio["array"].astype(np.float32)
            sampling_rate = audio["sampling_rate"]
            
            # --- Pre-processing: Resample to 16kHz and ensure mono ---
            # NeMo models expect 16kHz
            if sampling_rate != 16000:
                audio_array = librosa.resample(y=audio_array, orig_sr=sampling_rate, target_sr=16000)
                sampling_rate = 16000
            
            if audio_array.ndim > 1:
                audio_array = np.mean(audio_array, axis=1)
            # --------------------------------------------------------
            
            transcription = ""
            
            # --- Transcription ---
            with torch.no_grad():
                if model_name.startswith("nvidia/"):
                    # NeMo's transcribe method can take a NumPy array
                    # We pass a list containing our single array
                    transcriptions = asr_model.transcribe([audio_array])
                    if transcriptions:
                        transcription = transcriptions[0].text
            # ---------------------
            
            # Apply normalization to both reference and prediction
            references.append(normalize_arabic_text(reference_text))
            predictions.append(normalize_arabic_text(transcription))
        
        # Calculate metrics
        wer_score = wer(references, predictions)
        cer_score = cer(references, predictions)
        
        exact_matches = sum(1 for ref, pred in zip(references, predictions) if ref.strip() == pred.strip())
        accuracy = exact_matches / len(references)
        
        return {
            "model": model_name,
            "dataset": dataset_config["name"],
            "wer": wer_score,
            "cer": cer_score,
            "accuracy": accuracy,
            "samples_tested": len(references),
            "references": references,
            "predictions": predictions
        }
    
    except Exception as e:
        print(f"Error with {model_name}: {str(e)[:200]}...")
        import traceback
        traceback.print_exc()
        return None

In [6]:
# Cell 5: Run evaluation
all_results = []

for dataset_name, dataset_config in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*60}")
    
    for model_name, model_path in MODELS.items():
        # Use the new general-purpose function
        result = evaluate_model(model_path, dataset_config, max_samples=None)
        
        if result is not None:
            all_results.append(result)
            print(f"--- Results for {model_name} ---")
            print(f"WER: {result['wer']:.3f}")
            print(f"CER: {result['cer']:.3f}")
            print("----------------------------------")
            


DATASET: arabic_egy

Testing nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0...


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

data/train-00000-of-00028.parquet:   0%|          | 0.00/475M [00:00<?, ?B/s]

data/train-00001-of-00028.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

... (45 outputs left out)

[NeMo W 2025-11-08 20:23:45 nemo_logging:405] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: No `cuda-python` module. Please do `pip install cuda-python>=12.3`


[NeMo I 2025-11-08 20:23:46 nemo_logging:393] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_ar_fastconformer_hybrid_large_pcd_v1.0/snapshots/7f32349d952f42a28dce979ba73270aa2bbdfa89/stt_ar_fastconformer_hybrid_large_pcd_v1.0.nemo.


Evaluating nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0:   0%|                  | 0/10361 [00:00<?, ?it/s]
Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.86s/it]
Evaluating nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0:   0%|       | 1/10361 [00:04<11:56:06,  4.15s/it]
Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 17.11it/s]

Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 19.57it/s]
Evaluating nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0:   0%|        | 3/10361 [00:04<3:16:57,  1.14s/it]
Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.73it/s]

Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 18.54it/s]
Evaluating nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0:   0%|        | 5/10361 [00:04<1:43:05,  

--- Results for nvidia-conformer ---
WER: 0.485
CER: 0.244
----------------------------------
